# 深度学习 HW03
## 卷积网络、残差网络、图像增广和目标检测

**学号：** 20234080314
**姓名：** 刘轩昂 
**Date:** 2026-06-07

---
# 2 卷积和池化层

## 2.1 理论计算题

输入图像大小为 $3 \times 32 \times 32$（通道数 $\times$ 高 $\times$ 宽），卷积层包含 16 个卷积核，每个卷积核大小为 $3 \times 5 \times 5$，填充 $p=2$，步幅 $s=2$。

### 1. 输出特征图尺寸

卷积输出的高和宽为：

$$H_{out}=W_{out}=\left\lfloor\frac{32+2\times2-5}{2}\right\rfloor+1=\lfloor 15.5\rfloor+1=16$$

输出通道数等于卷积核个数，即 16。因此输出特征图尺寸为：

$$\boxed{16 \times 16 \times 16}$$

### 2. 单个输出通道的一个像素值需要多少次乘法

一个输出像素由一个卷积核与输入局部窗口做点乘得到。局部窗口大小为：

$$3 \times 5 \times 5 = 75$$

所以单个输出通道的一个像素值需要：

$$\boxed{75\text{ 次乘法}}$$

如果同时计算加法，则还需要 $74$ 次加法。

## 2.2 编程题：手动实现二维最大池化

In [1]:
import numpy as np

def max_pool2d(x, kernel_size, stride=1, padding=0):
    """NumPy实现二维最大池化，支持输入形状为 HxW、CxHxW 或 NxCxHxW。"""
    x = np.asarray(x, dtype=float)
    original_ndim = x.ndim
    if original_ndim == 2:
        x = x[None, None, :, :]
    elif original_ndim == 3:
        x = x[None, :, :, :]
    elif original_ndim != 4:
        raise ValueError('输入必须是 HxW、CxHxW 或 NxCxHxW')

    if isinstance(kernel_size, int):
        kh = kw = kernel_size
    else:
        kh, kw = kernel_size
    if isinstance(stride, int):
        sh = sw = stride
    else:
        sh, sw = stride
    if isinstance(padding, int):
        ph = pw = padding
    else:
        ph, pw = padding

    x_pad = np.pad(
        x,
        pad_width=((0, 0), (0, 0), (ph, ph), (pw, pw)),
        mode='constant',
        constant_values=-np.inf,
    )
    n, c, h, w = x_pad.shape
    out_h = (h - kh) // sh + 1
    out_w = (w - kw) // sw + 1
    y = np.empty((n, c, out_h, out_w), dtype=x.dtype)

    for i in range(out_h):
        for j in range(out_w):
            window = x_pad[:, :, i * sh:i * sh + kh, j * sw:j * sw + kw]
            y[:, :, i, j] = window.max(axis=(2, 3))

    if original_ndim == 2:
        return y[0, 0]
    if original_ndim == 3:
        return y[0]
    return y

# 示例验证
x = np.array([[1, 2, 3, 4],
              [5, 6, 7, 8],
              [9, 10, 11, 12],
              [13, 14, 15, 16]])
print(max_pool2d(x, kernel_size=2, stride=2, padding=0))

[[ 6.  8.]
 [14. 16.]]


---
# 3 LeNet、AlexNet、VGG 和 NiN

## 3.1 理论计算题

假设输入和输出特征图通道数均为 $C$，且卷积层不带偏置。

### 1. 一个 $5 \times 5$ 卷积层的参数量

$$5 \times 5 \times C \times C = \boxed{25C^2}$$

### 2. 两个串联的 $3 \times 3$ 卷积层的总参数量

每个 $3 \times 3$ 卷积层参数量为：

$$3 \times 3 \times C \times C = 9C^2$$

两个串联后的总参数量为：

$$\boxed{18C^2}$$

因此，用两个 $3 \times 3$ 卷积替代一个 $5 \times 5$ 卷积时，感受野同为 $5 \times 5$，参数量从 $25C^2$ 降为 $18C^2$，并且中间多了一次非线性激活。

## 3.2 编程题：PyTorch 定义 NiN Block

In [2]:
import torch
from torch import nn

def nin_block(in_channels, out_channels, kernel_size, stride, padding):
    return nn.Sequential(
        nn.Conv2d(in_channels, out_channels, kernel_size, stride, padding),
        nn.ReLU(),
        nn.Conv2d(out_channels, out_channels, kernel_size=1),
        nn.ReLU(),
        nn.Conv2d(out_channels, out_channels, kernel_size=1),
        nn.ReLU(),
    )

block = nin_block(3, 16, kernel_size=5, stride=1, padding=2)
X = torch.randn(2, 3, 32, 32)
Y = block(X)
print(block)
print('输入形状:', tuple(X.shape))
print('输出形状:', tuple(Y.shape))

Sequential(
  (0): Conv2d(3, 16, kernel_size=(5, 5), stride=(1, 1), padding=(2, 2))
  (1): ReLU()
  (2): Conv2d(16, 16, kernel_size=(1, 1), stride=(1, 1))
  (3): ReLU()
  (4): Conv2d(16, 16, kernel_size=(1, 1), stride=(1, 1))
  (5): ReLU()
)
输入形状: (2, 3, 32, 32)
输出形状: (2, 16, 32, 32)


---
# 4 Inception、批量归一化和残差网络

## 4.1 理论计算题

小批量中同一通道、同一空间位置的 4 个样本输出为：

$$x_1=2,\quad x_2=4,\quad x_3=6,\quad x_4=8$$

Batch Normalization 参数为 $\gamma=2$，$\beta=1$，$\epsilon=0$。

均值：

$$\mu=\frac{2+4+6+8}{4}=5$$

方差：

$$\sigma^2=\frac{(2-5)^2+(4-5)^2+(6-5)^2+(8-5)^2}{4}=5$$

归一化并缩放平移：

$$y_i=\gamma\frac{x_i-\mu}{\sqrt{\sigma^2+\epsilon}}+\beta=2\frac{x_i-5}{\sqrt{5}}+1$$

所以：

$$y_1=1-\frac{6}{\sqrt5}\approx -1.6833$$
$$y_2=1-\frac{2}{\sqrt5}\approx 0.1056$$
$$y_3=1+\frac{2}{\sqrt5}\approx 1.8944$$
$$y_4=1+\frac{6}{\sqrt5}\approx 3.6833$$

## 4.2 编程题：自定义 Residual 残差块

In [3]:
class Residual(nn.Module):
    def __init__(self, in_channels, out_channels, use_1x1conv=False, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3,
                               padding=1, stride=stride)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3,
                               padding=1)
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.conv3 = None
        if use_1x1conv:
            self.conv3 = nn.Conv2d(in_channels, out_channels, kernel_size=1,
                                   stride=stride)
        self.relu = nn.ReLU(inplace=True)

    def forward(self, X):
        Y = self.relu(self.bn1(self.conv1(X)))
        Y = self.bn2(self.conv2(Y))
        if self.conv3 is not None:
            X = self.conv3(X)
        Y += X
        return self.relu(Y)

# 示例1：通道数和空间尺寸不变
blk1 = Residual(3, 3)
X1 = torch.randn(4, 3, 32, 32)
print('blk1输出形状:', tuple(blk1(X1).shape))

# 示例2：使用1x1卷积调整通道数和尺寸
blk2 = Residual(3, 16, use_1x1conv=True, stride=2)
X2 = torch.randn(4, 3, 32, 32)
print('blk2输出形状:', tuple(blk2(X2).shape))

blk1输出形状: (4, 3, 32, 32)
blk2输出形状: (4, 16, 16, 16)


---
# 5 图像增广、微调和样式迁移

## 5.1 理论计算题

### 1. 为什么底层特征提取层用较小学习率或冻结，而顶层输出层用较大学习率？

预训练模型的底层特征提取层通常已经学到了较通用的视觉特征，例如边缘、纹理、颜色和局部形状。这些特征在 ImageNet 和新目标数据集之间往往具有较强迁移性。如果对这些层使用过大的学习率，容易破坏已经学到的通用表示，导致训练不稳定或泛化能力下降。

顶层输出层与原任务类别强相关，换到新数据集后通常需要重新初始化，因此需要更大的学习率让它更快适应新类别。常见做法是：冻结底层或为底层设置较小学习率，为新初始化的分类头设置较大学习率。

### 2. 目标数据集非常小且与源数据集相似时，应采取什么微调策略防止过拟合？

可以采用较保守的微调策略：冻结大部分预训练特征层，只训练最后的分类层；或者只解冻靠近输出端的少数高层特征层，并使用较小学习率。除此之外，还可以配合数据增广、权重衰减、Dropout、早停法以及验证集监控，减少小数据集上的过拟合风险。

## 5.2 编程题：torchvision.transforms 图像增广管道

In [4]:
from torchvision import transforms

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.08, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.5, contrast=0.5, saturation=0.5),
    transforms.ToTensor(),
])

print(train_transform)

Compose(
    RandomResizedCrop(size=(224, 224), scale=(0.08, 1.0), ratio=(0.75, 1.3333), interpolation=bilinear, antialias=True)
    RandomHorizontalFlip(p=0.5)
    ColorJitter(brightness=(0.5, 1.5), contrast=(0.5, 1.5), saturation=(0.5, 1.5), hue=None)
    ToTensor()
)


---
# 6 目标检测、计算机视觉训练技巧

## 6.1 理论计算题

真实框：$A=[10,10,50,50]$，预测框：$B=[30,30,70,70]$。

交集左上角为 $(30,30)$，右下角为 $(50,50)$，所以交集面积为：

$$S_{A\cap B}=(50-30)(50-30)=400$$

两个框的面积分别为：

$$S_A=(50-10)(50-10)=1600$$

$$S_B=(70-30)(70-30)=1600$$

并集面积为：

$$S_{A\cup B}=1600+1600-400=2800$$

IoU 为：

$$IoU=\frac{S_{A\cap B}}{S_{A\cup B}}=\frac{400}{2800}=\boxed{\frac17\approx 0.1429}$$

## 6.2 编程题：Label Smoothing 交叉熵损失

In [5]:
import torch.nn.functional as F

def smooth_one_hot(targets, num_classes, epsilon=0.1):
    """将类别标签转换为Label Smoothing后的目标分布。"""
    if not 0 <= epsilon < 1:
        raise ValueError('epsilon必须满足 0 <= epsilon < 1')
    targets = targets.long()
    off_value = epsilon / (num_classes - 1)
    on_value = 1.0 - epsilon
    y = torch.full((targets.size(0), num_classes), off_value,
                   device=targets.device, dtype=torch.float32)
    y.scatter_(1, targets.unsqueeze(1), on_value)
    return y

def label_smoothing_cross_entropy(logits, targets, epsilon=0.1):
    """计算Label Smoothing后的多分类交叉熵。"""
    num_classes = logits.size(1)
    smoothed = smooth_one_hot(targets, num_classes, epsilon)
    log_probs = F.log_softmax(logits, dim=1)
    return -(smoothed * log_probs).sum(dim=1).mean()

# 示例验证
logits = torch.tensor([[2.0, 0.5, -1.0],
                       [0.1, 1.2, 0.3]])
targets = torch.tensor([0, 2])
smoothed = smooth_one_hot(targets, num_classes=3, epsilon=0.1)
loss = label_smoothing_cross_entropy(logits, targets, epsilon=0.1)

print('平滑后的标签:')
print(smoothed)
print('Label Smoothing CE:', float(loss))

平滑后的标签:
tensor([[0.9000, 0.0500, 0.0500],
        [0.0500, 0.0500, 0.9000]])
Label Smoothing CE: 0.942437469959259


---
# 总结

本次作业完成了卷积输出尺寸、VGG 参数量、Batch Normalization、IoU 等理论计算，并手动实现了最大池化、NiN Block、Residual Block、图像增广管道和 Label Smoothing 交叉熵。